# บทที่ 4: ฟังก์ชันสูญเสียและการแพร่กระจายย้อนกลับ (Loss Function & Backpropagation)

ใน Notebook นี้ เราจะจำลองการทำงานของ Loss Functions ต่างๆ การคำนวณ Backpropagation ทีละขั้นตอน รวมถึงอธิบายการทำงานของ Optimizers เพื่อให้เห็นภาพรวมเชิงปฏิบัติการตามทฤษฎีในหนังสือ


## 1. นำเข้าไลบรารีที่จำเป็น (Import Libraries)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# ติดตั้งฟอนต์ภาษาไทยสำหรับ Google Colab
import subprocess, glob
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-tlwg-garuda'], 
               capture_output=True)

# ลงทะเบียนฟอนต์โดยตรง
from matplotlib.font_manager import fontManager
for font_file in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fontManager.addfont(font_file)

# ตั้งค่า Seaborn theme และฟอนต์ภาษาไทย
sns.set_theme(style='whitegrid', font='Garuda')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
%config InlineBackend.figure_format = 'retina'

## 2. ฟังก์ชันสูญเสีย (Loss Functions)
หัวข้อ 4.1 และ 4.2: การแปลงทฤษฎีเป็นโค้ด Python สำหรับ Mean Squared Error (MSE) และ Cross-Entropy


In [ ]:
def mse_loss(y_true, y_pred):
    """
    Mean Squared Error สำหรับปัญหา Regression
    """
    return np.mean((y_true - y_pred) ** 2)

def binary_cross_entropy(y_true, y_pred, epsilon=1e-15):
    """
    Binary Cross-Entropy สำหรับปัญหา Classification 2 คลาส
    (มีการบวกค่า epsilon เล็กๆ เพื่อป้องกันการหา log(0))
    """
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

def categorical_cross_entropy(y_true, y_pred, epsilon=1e-15):
    """
    Categorical Cross-Entropy สำหรับปัญหาหลายคลาส (Multi-class)
    """
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.sum(y_true * np.log(y_pred)) / y_pred.shape[0]

# --- ตัวอย่างการทดสอบฟังก์ชัน ---
y_true_reg = np.array([100.0, 150.0, 200.0])
y_pred_reg = np.array([110.0, 145.0, 190.0])
print(f"MSE Loss: {mse_loss(y_true_reg, y_pred_reg):.4f}")

y_true_bin = np.array([1, 0, 1])
y_pred_bin = np.array([0.9, 0.1, 0.8])
print(f"Binary Cross-Entropy Loss: {binary_cross_entropy(y_true_bin, y_pred_bin):.4f}")


## 3. กฎลูกโซ่และการแพร่กระจายย้อนกลับ (Chain Rule & Backpropagation)
หัวข้อ 4.3: จำลอง Forward Pass และ Backward Pass ด้วยหลักการหาอนุพันธ์ (Derivatives)


In [ ]:
# ฟังก์ชันกระตุ้นและอนุพันธ์ของฟังก์ชันกระตุ้น (Sigmoid)
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    # อนุพันธ์ของ Sigmoid คือ sigmoid(x) * (1 - sigmoid(x))
    sx = sigmoid(x)
    return sx * (1 - sx)

# กำหนดตัวแปรและน้ำหนักเริ่มต้นตัวอย่าง (จากหัวข้อ 4.7 ตัวอย่างการคำนวณแบบละเอียด)
x = np.array([[1.0], [2.0]]) # input
y = np.array([[1.0]])        # target

W1 = np.array([[0.5, 0.3], [0.2, 0.4]])
b1 = np.array([[0.1], [0.2]])

W2 = np.array([[0.6, 0.8]])
b2 = np.array([[0.3]])

learning_rate = 0.1

print("--- FORWARD PASS ---")
# Layer 1 (Hidden Layer)
z1 = np.dot(W1, x) + b1
a1 = sigmoid(z1)
print(f"Output ของ Hidden Layer (a1):\n{a1}")

# Layer 2 (Output Layer)
z2 = np.dot(W2, a1) + b2
y_hat = sigmoid(z2)
print(f"ผลทำนาย (y_hat): {y_hat[0][0]:.4f}")

# Loss (MSE)
loss = 0.5 * (y - y_hat)**2
print(f"Loss (MSE): {loss[0][0]:.4f}")

print("\n--- BACKWARD PASS ---")
# 1. Gradient ของ Output Layer (เทียบกับ z2)
dz2 = -(y - y_hat) * sigmoid_derivative(z2)

# Gradient ของ Parameters ฝั่ง Output (W2, b2)
dW2 = np.dot(dz2, a1.T)
db2 = dz2

# 2. Gradient ของ Hidden Layer (เทียบกับ z1)
da1 = np.dot(W2.T, dz2)
dz1 = da1 * sigmoid_derivative(z1)

# Gradient ของ Parameters ฝั่ง Hidden (W1, b1)
dW1 = np.dot(dz1, x.T)
db1 = dz1

# 3. อัปเดตพารามิเตอร์ (Gradient Descent แบบปกติตามหัวข้อ 4.4)
W1_new = W1 - learning_rate * dW1
b1_new = b1 - learning_rate * db1
W2_new = W2 - learning_rate * dW2
b2_new = b2 - learning_rate * db2

print("W1 ถูกอัปเดตเป็น:\n", W1_new)
print("W2 ถูกอัปเดตเป็น:\n", W2_new)


## 4. ตัวรับค่าพื้นฐาน (Basic Optimizers)
หัวข้อ 4.4: เปรียบเทียบ SGD, Momentum และอธิบายความแตกต่าง


In [ ]:
# สร้างพื้นผิวกราฟจำลอง Loss surface แบบง่ายๆ (พาราโบลา 1 มิติ) สำหรับสาธิต Optimizers
def loss_func(w):
    return w**2

def gradient(w):
    return 2*w

# กำหนดสภาวะเริ่มต้น
w_init = 10.0
learning_rate_demo = 0.1
epochs = 20

v_momentum = 0.0
beta = 0.9

w_sgd = w_init
w_momentum = w_init

history_sgd = [w_sgd]
history_momentum = [w_momentum]

for _ in range(epochs):
    # Standard SGD
    grad_sgd = gradient(w_sgd)
    w_sgd = w_sgd - learning_rate_demo * grad_sgd
    history_sgd.append(w_sgd)
    
    # Momentum
    grad_mom = gradient(w_momentum)
    v_momentum = beta * v_momentum + (1 - beta) * grad_mom # รูปแบบ Exponential Moving Average
    w_momentum = w_momentum - learning_rate_demo * v_momentum
    history_momentum.append(w_momentum)

plt.figure(figsize=(10, 5))
plt.plot(history_sgd, label='SGD', marker='o')
plt.plot(history_momentum, label='SGD with Momentum', marker='x')
plt.title("เปรียบเทียบการลู่เข้า (Convergence) ระหว่าง SGD และ Momentum")
plt.xlabel("Epochs")
plt.ylabel("Weight Value (Target = 0)")
plt.axhline(0, color='red', linestyle='--', label='Global Minimum')
plt.legend()
plt.show()

## 5. อาดัม (Adam: Adaptive Moment Estimation)
หัวข้อ 4.4.2: กลไกของ Adam ที่รวม Momentum และ Adaptive Learning Rate ไว้ด้วยกัน


In [ ]:
# สาธิตการทำงานของ Adam Optimizer ตัวหลัก
w_adam = w_init
m_t = 0.0
v_t = 0.0
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8
lr_adam = 0.5 # ใช้ LR สูงๆ เพื่อให้เห็นผลลัพธ์ชัดบนโจทย์ตัวอย่าง

history_adam = [w_adam]

for t in range(1, epochs + 1):
    grad = gradient(w_adam)
    
    # คำนวณ Moving Averages
    m_t = beta1 * m_t + (1 - beta1) * grad
    v_t = beta2 * v_t + (1 - beta2) * (grad**2)
    
    # Bias Correction (หัวใจสำคัญตอนเริ่มต้นเรียนรู้)
    m_t_hat = m_t / (1 - beta1**t)
    v_t_hat = v_t / (1 - beta2**t)
    
    # การอัปเดตน้ำหนัก
    w_adam = w_adam - lr_adam * m_t_hat / (np.sqrt(v_t_hat) + epsilon)
    history_adam.append(w_adam)

plt.figure(figsize=(10, 5))
plt.plot(history_adam, color='purple', label='Adam (LR=0.5)', marker='s')
plt.title("พฤติกรรมการลู่เข้าของ Adam Optimizer")
plt.xlabel("Epochs")
plt.ylabel("Weight Value")
plt.axhline(0, color='red', linestyle='--')
plt.legend()
plt.show()

## 6. ปัญหาเกี่ยวกับขนาดการไล่ระดับ (Vanishing / Exploding Gradient)
หัวข้อ 4.5: สาธิตสิ่งที่เกิดขึ้นเมื่อ Gradient ผ่าน Layer จำนวนมากด้วย Sigmoid เทียบกับ ReLU


In [ ]:
def relu_derivative(x):
    return np.where(x > 0, 1, 0)

# จำลอง Gradient เริ่มต้นที่ไหลมาจาก Output (ตีกรรม 1.0)
initial_grad = 1.0

# สุ่มข้อมูล Input ระหว่างช่วงต่างๆ
inputs_sigmoid = np.random.uniform(-3, 3, 10)
inputs_relu = np.random.uniform(-3, 3, 10)

grad_flow_sigmoid = [initial_grad]
grad_flow_relu = [initial_grad]

curr_grad_sig = initial_grad
curr_grad_relu = initial_grad

print("จำลองการไหลย้อนของ Gradient ผ่าน 10 Layers (ซิมิวเลเตอร์):")
for i in range(10):
    curr_grad_sig = curr_grad_sig * sigmoid_derivative(inputs_sigmoid[i])
    grad_flow_sigmoid.append(curr_grad_sig)
    
    curr_grad_relu = curr_grad_relu * relu_derivative(inputs_relu[i])
    grad_flow_relu.append(curr_grad_relu)

plt.figure(figsize=(10, 5))
plt.plot(grad_flow_sigmoid, label='Gradient Flow (Sigmoid)', marker='v')
plt.plot(grad_flow_relu, label='Gradient Flow (ReLU)', marker='^')
plt.yscale('log')
plt.title("Vanishing Gradient Problem: Sigmoid vs ReLU (แกน Y แบบ Log Scale)")
plt.xlabel("Layers Traversed (ย้อนกลับจาก Output สู่ Input)")
plt.ylabel("Gradient Magnitude (Log)")
plt.legend()
plt.show()

print("บทสรุป: จะสังเกตได้ว่า Gradient ฝั่ง Sigmoid หดตัวหายไปเข้าใกล้ 0 อย่างรวดเร็วมาก (ลู่ลงด้านล่างของกราฟ Log) เพราะอนุพันธ์สูงสุดไม่เกิน 0.25 ในขณะที่ ReLU รักษาสัญญาณได้ดีกว่า")

## บทสรุป
Notebook นี้เป็นส่วนขยายของบทที่ 4 ผู้อ่านสามารถเปลี่ยนค่าตัวแปร เช่น `learning_rate`, `epochs` หรือโครงสร้าง `W1`, `W2` เพื่อสังเกตผลที่ตามมา และนำไปปรับใช้ในอัลกอริทึมจริงต่อไปได้
